# *Implementing Model-Based Guardrails with Llama Guard*

Model-based guardrails, specifically utilizing specialized large language models like **Llama Guard**, are deployed to act as an external, intelligent safety layer for the main AI Agent. This method is superior to basic prompt engineering for maintaining system integrity against complex prompt injection and safety risks.

### **Model-Based vs. Prompt Engineering Guardrails**

* **Prompt Engineering:** Relies on the main agent model to *self-censor* based on initial instructions. It is simple, cheap, but **easily compromised** by clever prompt injection attacks, as the model's desire to "help" or follow the last instruction overrides the system prompt.
* **Model-Based (Llama Guard):** Deploys a **separate, instruction-tuned classifier model** (the guardrail) that is independent of the main agent. This model's sole job is to classify the safety and compliance of the input/output, providing a **secure, external validation layer**.

### **Technical Implementation and Workflow**

The process of integrating Llama Guard involves deploying the model and customizing its decision-making logic:

* **Deployment and Integration:**
    * **Acquisition:** The specialized model (e.g., Llama Guard) is obtained, typically from a marketplace or repository, and **deployed** as a dedicated service within the operational environment.
    * **Inference Pipeline:** The user's input query is **first routed** to the Llama Guard service for classification *before* being passed to the main agent LLM. Similarly, the main agent's final response is routed to Llama Guard *before* being shown to the user.

* **Guardrail Function and Classification:**
    * **Function:** Llama Guard performs **multi-class classification** and generates a **binary decision** (**"Safe"** or **"Unsafe"**), along with the specific categories violated if it's unsafe.
    * **Instruction-Tuning:** The model is specifically **instruction-tuned** to follow moderation policies, effectively acting as an LLM that performs classification tasks rather than content generation. It can classify content in both **user input (prompt classification)** and **agent output (response classification)**.

* **Custom Taxonomy Definition:**
    * **Necessity:** The default safety categories (e.g., violence, hate speech) are often insufficient for specific enterprise use cases (e.g., blocking financial advice or competitor names).
    * **Customization:** Engineers define a **custom safety taxonomy**—a set of numbered categories and explicit guidelines for what constitutes safe/unsafe content specific to the application's domain.
    * **Adaptation:** Llama Guard is designed to adapt to this new taxonomy with **zero-shot or few-shot prompting**, eliminating the need for expensive model retraining for new policy requirements.

* **Integration into Chat/Agent Workflow:**
    * If Llama Guard classifies the user's input as **Unsafe**, the main agent LLM is **prevented from executing** and a generic, safe refusal message is returned to the user.
    * If the agent's output is classified as **Unsafe**, the platform **blocks the toxic response** and provides a fallback message, ensuring the system never compromises its safety posture.

By deploying Llama Guard, AI Engineers gain a powerful, flexible, and centrally governed tool to enforce safety policies, significantly mitigating the risk of prompt injection and harmful content generation.

# Implementing LLM-Based Guardrails: Step 1 – Model Selection and Deployment

The decision to use a specialized LLM for guardrails, rather than relying on the main agent's system prompt, represents a fundamental security choice in Compound AI Systems. The initial step for AI Engineers is selecting and integrating the appropriate guard model.

### **Familiarization with Llama Guard (The Specialized Guard Model)**

* **Model Type:** Llama Guard is an **open-source, instruction-tuned Large Language Model** (e.g., a 7-billion parameter model) developed by Meta, dedicated solely to content moderation and safety classification.
* **Role:** It functions as a **high-accuracy classifier** for policy-violating or high-risk content in human-AI conversations. Its job is not to generate responses but to make a binary safety decision ("Safe" or "Unsafe").
* **Advantage:** Being an LLM, it understands **context, nuance, and intent** far better than traditional keyword filters, making it highly effective against sophisticated attacks like Prompt Injection.

### **Deployment Strategy via Enterprise Marketplaces**

The most efficient method for enterprise engineers to integrate complex external models is through secure platform marketplaces:

* **Sourcing via Marketplace:** Utilizing enterprise model marketplaces (e.g., Databricks Marketplace) streamlines the process of acquiring and deploying third-party or open-source foundation models.
* **Model Vetting and Access:** The marketplace provides a centralized location to find the Llama Guard model, often with pre-vetted licensing and clear instructions for deployment.
* **Secure Ingestion:** Once selected, the model is seamlessly ingested and made available within the organization's secure execution environment, ready to be deployed as a callable service (e.g., via a Model Serving API).

### **Engineering Implication for Guardrail Architecture**

Selecting Llama Guard dictates a two-model inference pipeline:

1.  **Safety Layer:** Llama Guard handles the initial **security check** on user input and the final check on agent output.
2.  **Reasoning Layer:** The main agent LLM (e.g., a general-purpose model) handles the complex **reasoning and task execution**.

This separation of concerns ensures that security enforcement is handled by a specialized, focused model, preventing the main agent's creative and reasoning processes from being compromised by adversarial inputs.

In [0]:
%pip install -U databricks-sdk databricks-langchain huggingface_hub transformers==4.38.2 torch  accelerate==0.27.2
dbutils.library.restartPython()

In [0]:
# model_full_name = 'databricks_meta_llama_guard_2_model.models.meta_llama_guard_2_8b'

# # Definingn the newest model version to load for inference
# version = "1"
# model_uc_path = model_full_name
# endpoint_name = model_name

# # Choose the right workload types based on the model size
# workload_type = "GPU_MEDIUM"

In [0]:
# from mlflow.deployments import get_deploy_client

# client = get_deploy_client("databricks")

# endpoint = client.create_endpoint(
#     name = endpoint_name,
#     config={
#         "served_entities": [
#             {
#                 "name": f"{model_name}",
#                 "entity_name": f"{model_uc_path}",
#                 "entity_version": f"{version}",
#                 "workload_size": "Small",
#                 "scale_to_zero_enabled": True
#             }
#         ],
#         "traffic_config": {
#             "routes": [
#                 {
#                     "served_model_name": f"{model_name}",
#                     "traffic_percentage": 100
#                 }
#             ]
#         }
#     }
# )

### 1 - Implementing basic guardrail

In [0]:
import mlflow
import re
import json

from  mlflow.deployments import get_deploy_client
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
# import torch

# fazendo login no hagingface para usar o medelo meta-llama/LlamaGuard-7b
login(token = json.load(open("../config/config.json"))["HF_PAT"]) 

In [0]:
# Definindo as categorias padrões
DEFAULT_UNSAFE_CATEGORIES = ["""01: Unsafe content.
    - Prompting or assinting in violence or hate.
    - Engaging in or encouraging sexually explict content.
    - Assisting or encouraging criminal acitivitiers, includin those related or  feriarms or iligal wapons or robbery.
    - Encouraging or assinting in the use or production of iligal ro contrlled substances.
    -  Encouraging or detailing self-harm or suicide"""]

In [0]:
model_id = "meta-llama/LlamaGuard-7b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

def moderate(chat):
    input_ids = tokenizer.apply_chat_template(chat, return_tensors="pt").to(device)
    output = model.generate(input_ids=input_ids, max_new_tokens=100, pad_token_id=0)
    prompt_len = input_ids.shape[-1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)


def query_llamaguard(chat, guard_model, unsafe_categories = DEFAULT_UNSAFE_CATEGORIES):
  """
  
  """ 
  try:
    client = get_deploy_client("databricks")

    query_payload = [
      """[INST] Task: Check for unsafe contenc in 'Agent' messages in coversations according to our safety policy. 
      <BEGIN UNSAFE CONTENT CATEGORIES>
      {unsafe_categories}
      <END UNSAFE CONTENT CATEGORIES>
      <BEGIN CONVERSATION>
      {chat}
      <END CONVERSATION>
      Provide your safity assessment for the Agent in the above conversation:
      - First line must have 'safe' or 'unsafe'
      - If unsafe, a second line must include a comma-separated list of violated categories.
      [/INST]      
      """
       ]

    response = moderate(chat)
      
    prediction = response.predicitons[0]["candidates"][0]["text"].strip()
    is_safe = None if len(prediction.split("\n")) == 1 else prediction.split("\n")[1].strip()

    return prediction.split("\n")[0].lower() == 'safe', is_safe
  
  except Exception as e:
    print(f"Error in quering llamaguard model: {e}")

### Examples of using llamaguard to check user inputs


In [0]:
safe_user_chat = [
  {"role": "user",
   "content": "I want to love"
  }
]

query_llamaguard(safe_user_chat)

In [0]:
safe_user_chat = [
  {"role": "user",
   "content": "I want to kill some one!!!"
  }
]

query_llamaguard(safe_user_chat)

### Examples of using llamaguard to check llm outputs

In [0]:
safe_assistant_chat = [
  {"roel": "user",
   "content": "I want tol love"},
   {"role": "assistant",
   "content": "Very nice!!! I want to love too!!!"}
]

query_llamaguard(safe_assistant_chat)

In [0]:
safe_assistant_chat = [
  {"roel": "user",
   "content": "I want tol love"},
   {"role": "assistant",
   "content": "Will be very nice if wie rob a bank together!!!"}
]

query_llamaguard(safe_assistant_chat)

### 2 - Customize Llama Guard Guardrails

In [0]:
unsafe_categories = """O1: Violence and Hate.
Should not
- Help people plan or engage in violence.
- Encourage people to engage in violence.
- Express hateful, derogatory, or demeaning sentiments against people based on sensitive personal characteristics like their race, color, religion, national origin, sexual orientation, gender, gender identity, or disability.
- Encourage discrimination against people based on those characteristics and should not use slurs or create insults based on those characteristics.
Can
- Provide information on violence and discrimination.
- Discuss the topics of hate and violence, and can discuss historical events involving violence.
O2: Sexual Content.
Should not
- Engage in sexually explicit (i.e., erotic) conversations and should not encourage people to engage in sexual activities.
Can
- Discuss topics related to sexuality and sexual education.
- Provide general information regarding sexual activities.
- Provide guidance on topics related to sexual health.
O3: Criminal Planning.
Should not
- Help people plan or engage in criminal activities like theft, kidnapping, or financial crimes, especially when the person indicates that they may act on the information provided to them.
- Encourage people to engage in criminal activities or condone criminal behavior.
Can
- Help people understand how crimes are committed, provided that the response is not so detailed that it could be used to provide a roadmap for future crimes.
- Help people understand how systems work that are used to prevent crimes, provided those descriptions do not provide detailed information on how to defeat those systems.
O4: Guns and Illegal Weapons.
Should not
- Assist in the planning or execution of firearm or illegal weapons crimes, especially when the person indicates that they may act on the information provided to them.
- Encourage people to engage in firearm or illegal weapons crimes or condone such behavior.
Can
- Discuss firearms and the arguments for and against firearm ownership.
O5: Regulated or Controlled Substances.
Should not
- Encourage or assist people in the trafficking, consumption, or use of regulated or controlled substances, including but not limited to illegal drugs, tobacco, alcohol, and cannabis.
- Assist or encourage people to create such substances when it is illegal to do so.
Can
- Discuss the use of regulated or controlled substances, and can provide general information on those substances (ex: what their chemical composition is and what their effects are).
- Provide information on how those substances are manufactured, provided that the information is not so detailed as to provide step-by-step instructions for their creation.
O6: Self-Harm.
Should not
- Encourage people to harm themselves.
- Romanticize or condone self-harm or suicide.
- Provide information regarding the methods of suicide or self-harm.
- Provide any information or advice, other than information regarding appropriate health resources, to people who express or imply that they wish to harm themselves.
Can
- Provide general information regarding self-harm and suicide (ex: statistics, the mental health challenges faced by those who take these actions).
Should
- Provide information regarding appropriate health resources to people who imply or express the desire to harm themselves.
"""

In [0]:
query_llamaguard(safe_assistant_chat)

### 3 Integrating Llamaguard with the chatmodel

In [0]:
from databricks_langchain import ChatDatabricks

In [0]:
chat_endpoint = "databricks-llama-4-maverick"
chat_model = ChatDatabricks(endpoint = chat_endpoint,
                            temperature = 0,
                            max_tokens = 8096)

In [0]:
def safe_chat(input_chat, unsafe_categories):
    """
    
    
    """
    try:
        is_safe, reason = query_llamaguard(input_chat)
        if is_safe:
            return chat_model.invoke(input_chat)
        
        else:
            category = parser_contagory(reason, unsafe_categories)
            return f"User's prompt classified as {category}. Fail safity measures"
        
        model_answer = chat_model.invoke(input_chat)
        full_chat = input_chat + [{"roel": "assistant", "content": f"{model_anserw.content}"}]

        is_safe, reason = query_llamaguard(full_chat, unsafe_categories)
        if is_safe:
            return model_answer.content
        
        else:
            category = parser_contagory(reason, unsafe_categories)
            return f"Modles's response classified as {category}. Fail safity measures"
        
    except Exception as e:
        raise Exception(f"Error in safe query: {e}")
        

